# RNN 영화리뷰 긍정 부정

# Basic setting

### Measuring execution time

In [80]:
!pip install --q ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.3 MB/s eta 0:00:00
time: 301 µs (started: 2025-01-24 02:24:34 +00:00)


### Library

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers
import os
import requests
import zipfile
import io

from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.metrics import classification_report, confusion_matrix


time: 860 µs (started: 2025-01-24 02:24:34 +00:00)


# Data collection

### Load data with zip file from URL

In [82]:
# Zip file URL
zip_url = 'https://raw.githubusercontent.com/20161609/data_box/main/movie_review.zip'
response = requests.get(zip_url)
if response.status_code == 200:
    zip_data = io.BytesIO(response.content)  # Processing in memory
    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        zip_ref.extractall('/content')  # Decompress in memory

    data_root = '/content/movie_review'
    train_dir = data_root + '/train'
    test_dir = data_root + '/test'
else:
    raise Exception(f"Failed to download data. Status code: {response.status_code}")

train = pd.read_csv('/content/trainData.tsv', delimiter='\t')
test = pd.read_csv('/content/testData.tsv', delimiter='\t')

train.shape, test.shape

((25000, 3), (25000, 2))

time: 2.25 s (started: 2025-01-24 02:24:34 +00:00)


In [83]:
train.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


time: 465 ms (started: 2025-01-24 02:24:36 +00:00)


In [84]:
train.iloc[0].review[:40:]

'With all this stuff going down at the mo'

time: 4.28 ms (started: 2025-01-24 02:24:37 +00:00)


# Data preprocessing

### Delete HTML tag

In [85]:
from bs4 import BeautifulSoup

bs = BeautifulSoup(train.iloc[0].review, 'html.parser')
bs.text

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.The actual feature film bit when it finally starts is only on for 20 mi

time: 5.95 ms (started: 2025-01-24 02:24:37 +00:00)


### Delete numbers and symbols - regular expression

In [86]:
import re

cleaned = re.sub('[^a-zA-Z]', ' ', bs.text)
cleaned

'With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle messages about MJ s feeling towards the press and also the obvious message of drugs are bad m kay Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring  Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him The actual feature film bit when it finally starts is only on for    mi

time: 5.44 ms (started: 2025-01-24 02:24:37 +00:00)


In [87]:
### Uppercase -> Lowercase

time: 388 µs (started: 2025-01-24 02:24:37 +00:00)


In [88]:
leaned = cleaned.lower()
cleaned

'With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle messages about MJ s feeling towards the press and also the obvious message of drugs are bad m kay Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring  Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him The actual feature film bit when it finally starts is only on for    mi

time: 4.31 ms (started: 2025-01-24 02:24:37 +00:00)


### stopwords

In [89]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

time: 9.05 ms (started: 2025-01-24 02:24:37 +00:00)


### Retrieve stopwords.

In [90]:
from nltk.corpus import stopwords

eng_stopwords = stopwords.words('english')
eng_stopwords[:5:]

['i', 'me', 'my', 'myself', 'we']

time: 3.48 ms (started: 2025-01-24 02:24:37 +00:00)


### Text preprocessing

In [91]:
def preprocess(sentence):
    soup = BeautifulSoup(sentence, 'html.parser')
    cleaned = re.sub('[^a-zA-Z]', ' ', soup.text)
    cleaned = cleaned.lower()
    cleaned = [word for word in cleaned.split() if word not in eng_stopwords]
    return ' '.join(cleaned)

preprocess(train.iloc[0].review)

train_clean = train['review'].apply(preprocess)
train_clean.head()

<ipython-input-91-2ba130426d2b>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(sentence, 'html.parser')


,review
0,stuff going moment mj started listening music ...
1,classic war worlds timothy hines entertaining ...
2,film starts manager nicholas bell giving welco...
3,must assumed praised film greatest filmed oper...
4,superbly trashy wondrously unpretentious explo...


time: 53.1 s (started: 2025-01-24 02:24:37 +00:00)


# Data preparation

## Tokenizer

In [92]:
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(train_clean)

len(tokenizer.word_index), tokenizer.word_index['odd']

(74066, 874)

time: 2.6 s (started: 2025-01-24 02:25:30 +00:00)


### Split: train and test

In [93]:
train_lable = train['sentiment']
train_lable.head()

,sentiment
0,1
1,1
2,0
3,0
4,1


time: 8.28 ms (started: 2025-01-24 02:25:33 +00:00)


In [94]:
from sklearn.model_selection import train_test_split

train_sentence, val_sentence, train_label, val_label = train_test_split(train_clean, train_lable, test_size=0.2, random_state=42)
train_sentence.shape, val_sentence.shape

((20000,), (5000,))

time: 12.2 ms (started: 2025-01-24 02:25:33 +00:00)


### Sequence

In [95]:
train_sequence = tokenizer.texts_to_sequences(train_sentence)
val_sequence = tokenizer.texts_to_sequences(val_sentence)

print(train_sequence[0])

[2, 887, 842, 821, 3003, 14250, 1710, 3772, 24779, 1175, 3, 3297, 1485, 8391, 1710, 3772, 4, 1086, 1052, 168, 26856, 869, 29536, 7405, 160, 1167, 177, 3134, 589, 3772, 1861, 80, 819, 15, 726, 3, 375, 3772, 10390, 3226, 16, 357, 13, 842, 15, 134, 124, 4, 713, 1160, 16758, 3750, 1110, 398, 2, 99, 173, 4304, 178, 155, 72211, 685, 72212, 1673, 16, 118, 2316, 8277, 1200, 53, 2209, 1861, 7592, 34495, 72213, 482, 72214, 1144, 246, 2517, 2191, 2808, 22, 366, 246, 2497, 53, 6810, 205, 138, 839, 87, 17142, 3750, 1110, 16, 118, 6049, 3772, 3911, 4265, 1411, 450, 415, 4265, 172, 1374, 122, 363, 118, 1094, 112, 246, 2265, 1364, 1580, 1797, 16, 101, 32, 170, 106, 30, 614]
time: 1.34 s (started: 2025-01-24 02:25:33 +00:00)


### Pedding

In [96]:
from keras.preprocessing.sequence import pad_sequences

train_padded = pad_sequences(train_sequence,
              maxlen=150,
              padding='pre',
              truncating='pre')

val_padded = pad_sequences(val_sequence,
              maxlen=150,
              padding='pre',
              truncating='pre')

train_padded.shape

(20000, 150)

time: 210 ms (started: 2025-01-24 02:25:34 +00:00)


# Model

### Create model

In [97]:
train_label = train_label.to_numpy()
val_label = val_label.to_numpy()
train_label

EMBEDDING_DIM = 300
VAOCA_SIZE = len(tokenizer.word_index)+1
VAOCA_SIZE

model = keras.Sequential([
    layers.Embedding(VAOCA_SIZE, EMBEDDING_DIM, input_length=150),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(128),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

time: 59 ms (started: 2025-01-24 02:25:34 +00:00)


### Compile model

In [98]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

time: 10.8 ms (started: 2025-01-24 02:25:34 +00:00)


### Learning and get history

In [99]:
EPOCHS = 1
BATCH_SIZE = 32

history = model.fit(
    train_padded, train_label,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE,
    # validation_split = 0.2
    validation_data=(val_padded, val_label)
)

625/625 ━━━━━━━━━━━━━━━━━━━━ 640s 1s/step - accuracy: 0.7553 - loss: 0.4825 - val_accuracy: 0.8684 - val_loss: 0.3114
time: 10min 39s (started: 2025-01-24 02:25:34 +00:00)


# Estimate

### Test data

In [100]:
test_clean = test['review'].apply(preprocess)
test_sequence = tokenizer.texts_to_sequences(test_clean)

<ipython-input-91-2ba130426d2b>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(sentence, 'html.parser')


time: 21.6 s (started: 2025-01-24 02:36:14 +00:00)


### Padding sequence

In [101]:
test_padded = pad_sequences(test_sequence, maxlen=150, padding='pre', truncating='pre')

time: 182 ms (started: 2025-01-24 02:36:36 +00:00)


### test lable

In [102]:
test_label = test['sentiment'].to_numpy()

KeyError: 'sentiment'

time: 116 ms (started: 2025-01-24 02:36:36 +00:00)


### Estimate model

In [ ]:
test_loss, test_accuracy = model.evaluate(test_padded, test_label, batch_size=BATCH_SIZE)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

### Predict

In [ ]:
predictions = model.predict(test_padded, batch_size=BATCH_SIZE)
predictions = [1 if p > 0.5 else 0 for p in predictions]

### Visualize the perfomance of model

In [ ]:
# Analize model
print("Classification Report:")
print(classification_report(test_label, predictions))

print("Confusion Matrix:")
print(confusion_matrix(test_label, predictions))

# Visualization
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(test_label, predictions), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()